# Detección de fraude con tarjetas: código seleccionado

Extracto de la sección de tarjetas del capstone supervisado, asignada a KFB. Se retiraron las salidas y el montaje de Google Drive. El archivo de transacciones se pasa mediante CREDIT_CARD_DATA_PATH y no se incluye aquí. El código conserva el carácter exploratorio del notebook original; no se ha vuelto a ejecutar para esta curaduría.


In [ ]:
import imblearn

import os
import pandas as pd

# Utilice dataset credit_card.txt
cc = pd.read_csv(os.environ['CREDIT_CARD_DATA_PATH'])
cc.head()
# Utilice dataset credit_card.txt

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

data = cc

summary = data.describe()
info = data.info()

class_counts = data['Class'].value_counts()

correlation_with_class = data.corr(numeric_only=True)['Class'].sort_values(ascending=False)

# Graficar distribución de clases
plt.figure(figsize=(6, 4))
sns.countplot(x='Class', data=data)
plt.title("Distribución de Clases")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.tight_layout()
plt.show()

class_counts, correlation_with_class

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = data.drop('Class', axis=1)  # Características
y = data['Class']  # Etiqueta

# Dividimos los datos en conjunto de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# utilizamos SMOTE para balancear las clases en el conjunto de entrenamiento
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Ver el balance antes y después del resampling
print(f'Antes del resampling: {y_train.value_counts()}')
print(f'Después del resampling: {y_train_resampled.value_counts()}')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Inicializamos el modelo de regresión logística
logreg = LogisticRegression(random_state=42, max_iter=10000)

# Entrenamos el modelo
logreg.fit(X_train_resampled, y_train_resampled)

# Hacemos predicciones en el conjunto de prueba
y_pred = logreg.predict(X_test)

In [ ]:
# Reporte de clasificación
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

# Matriz de confusión
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Alternativa: Usar under sampling
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_resampled, y_resampled = rus.fit_resample(X_train, y_train)

print(f"Clase 0 (no fraude): {sum(y_resampled == 0)}")
print(f"Clase 1 (fraude): {sum(y_resampled == 1)}")

logreg_rus = LogisticRegression(max_iter=50000, random_state=42)
logreg_rus.fit(X_resampled, y_resampled)

y_pred = logreg_rus.predict(X_test)

In [ ]:
# Reporte de clasificación
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))

# Matriz de confusión
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Inicializamos el modelo RandomForest
rf = RandomForestClassifier(random_state=42)

# Entrenamos el modelo usando el conjunto resampleado con SMOTE
rf.fit(X_train_resampled, y_train_resampled)

# Hacemos predicciones en el conjunto de prueba
y_pred_rf = rf.predict(X_test)

# Reporte de clasificación
print("Reporte de clasificación con SMOTE (oversampling):")
print(classification_report(y_test, y_pred_rf))

# Matriz de confusión
print("Matriz de confusión con SMOTE (oversampling):")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
# Inicializamos el modelo RandomForest
rf_rus = RandomForestClassifier(random_state=42)

# Entrenamos el modelo usando el conjunto resampleado con undersampling
rf_rus.fit(X_resampled, y_resampled)

# Hacemos predicciones en el conjunto de prueba
y_pred_rf_rus = rf_rus.predict(X_test)

# Reporte de clasificación
print("Reporte de clasificación con undersampling (RandomUnderSampler):")
print(classification_report(y_test, y_pred_rf_rus))

# Matriz de confusión
print("Matriz de confusión con undersampling (RandomUnderSampler):")
print(confusion_matrix(y_test, y_pred_rf_rus))

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# Inicializamos el modelo de Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)

# Entrenamos el modelo
gb.fit(X_train_resampled, y_train_resampled)

# Hacemos predicciones en el conjunto de prueba
y_pred_gb = gb.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Reporte de clasificación
print("Reporte de clasificación con Gradient Boosting:")
print(classification_report(y_test, y_pred_gb))

# Matriz de confusión
print("Matriz de confusión con Gradient Boosting:")
print(confusion_matrix(y_test, y_pred_gb))

In [ ]:
data = {
    'Modelo': ['RegresionLogistica', 'RandomForest', 'GradientBoosting'],
    'Verdaderos Negativos (No Fraude)': [56395, 56852, 56539],
    'Falsos Positivos (Fraude)': [469, 12, 325],
    'Falsos Negativos (No Fraude)': [9, 15, 10],
    'Verdaderos Positivos (Fraude)': [89, 83, 88]
}

df_comparativa = pd.DataFrame(data)
df_comparativa